# 도서관 WiFi QoE 예측 — 모델 비교

KT 오픈데이터 챌린지 3위 수상작의 모델링 과정입니다. 5분 간격 WiFi 로그로 QoE 지표를 만들고, 30분치 시퀀스로 5분 뒤 QoE를 예측합니다.

비교 대상: Persistence 기준선, GRU, CNN-LSTM, Transformer, LightGBM.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..')) if os.path.basename(os.getcwd()) == 'notebooks' else sys.path.append('ml')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. QoE 지표 계산

`ml/qoe_features.py`에 정의를 모아 두어 학습과 서빙이 같은 산식을 씁니다.

- 지연·지터·손실은 로그 정규화로 낮은 구간의 변화를 크게 봅니다. 핑이 10ms에서 40ms로 오를 때 체감 저하가 300ms에서 330ms로 오를 때보다 크기 때문입니다.
- 가중치는 지연 0.25, 다운로드 0.25, 손실 0.20, 지터 0.15, RSSI 0.10, 접속자 수 0.05입니다. 지연·손실 계열에 0.6을 몰아 체감 저하를 먼저 반영합니다.
- EWM(span=12, 약 1시간)으로 평활해 순간 튐을 걸러냅니다.

In [ ]:
from qoe_features import FEATURES, TIMESTEPS_DEFAULT, prepare, make_sequences

CSV = '../backend/db/dataset_plus.csv'
df = prepare(CSV)
print(df.shape)
df[['ping_ms', 'download_Mbps', 'packet_loss_rate', 'QoE_index', 'QoE_index_future']].head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['QoE_index'].values[:800], color='black', linewidth=1)
ax.set_title('QoE index (0 = 쾌적, 1 = 저하)')
ax.set_xlabel('time step (5min)')
ax.grid(alpha=0.3)
plt.show()

## 2. Persistence 기준선

"5분 뒤 QoE는 지금과 같다"고 답하는 모델입니다. QoE는 자기상관이 강해서 이 기준선을 넘지 못하면 딥러닝을 쓸 이유가 없습니다.

In [ ]:
from baseline import evaluate

base = evaluate(CSV)
base

## 3. 모델 학습·비교

전체 학습 루프는 `ml/train_models.py`에 있습니다. 노트북에서는 그대로 불러 실행합니다.

```bash
python ml/train_models.py --csv backend/db/dataset_plus.csv --plot
```

In [ ]:
from train_models import main
from argparse import Namespace

main(Namespace(csv=CSV, timesteps=TIMESTEPS_DEFAULT, outdir='../ml/figures', plot=True))

## 4. 결과 해석

- 회귀 지표(MSE/MAE)는 Persistence와 나란히 놓고 봐야 의미가 있습니다.
- 서비스에서 쓰는 값은 연속값이 아니라 Good/Moderate/Bad 3단계이므로, 분류 정확도와 macro F1을 함께 봅니다. 임계값은 학습 구간의 33/66 분위수로 잡습니다.
- 최종 서빙 모델은 GRU이며 `backend/models/gru_qoe.h5`, `scaler.pkl`, `thresholds.txt`로 저장해 FastAPI가 그대로 씁니다.